In [ ]:
#!/usr/bin/env python
# coding: utf-8

In [1]:
cd "D:\\brain age prep and meth"
myenv\\Scripts\\activate

SyntaxError: invalid syntax (256282756.py, line 1)

In [1]:
# -*- coding: utf-8 -*-
"""
Fixed preprocessing pipeline for SFCN brain-age model.

Folder structure expected:

T1/
├── OAS30001_MR_d0129/
│   ├── anat2/
│   │   ├── sub-OAS30001_ses-d0129_run-1_T1w.json
│   │   └── sub-OAS30001_ses-d0129_run-1_T1w.nii.gz
│   └── anat3/
│       └── ...
│
├── OAS30001_MR_d0757/
│   └── anat2/
│       └── ...
│
└── ...

The batch preprocessing searches recursively through all
subdirectories inside T1.
"""

import os
import glob
import ants
import antspynet
import numpy as np
import torchio as tio
import nibabel as nib


# =========================================================
# MNI152 brain template cache
# =========================================================

_MNI_BRAIN_CACHE = "mni152_brain_1mm.nii.gz"


def get_mni_brain_template():
    """
    Returns a skull-stripped 1mm MNI152 template.

    The template is generated once and cached to disk so that
    brain extraction does not need to be repeated for every scan.
    """

    if os.path.exists(_MNI_BRAIN_CACHE):
        return ants.image_read(_MNI_BRAIN_CACHE)

    print("Creating skull-stripped MNI152 template...")

    mni_head = ants.image_read(
        antspynet.get_antsxnet_data("mni152")
    )

    # brain_extraction() returns a probability image.
    mni_mask_prob = antspynet.brain_extraction(
        mni_head,
        modality="t1"
    )

    # Convert probability mask to binary mask.
    mni_mask_bin = ants.threshold_image(
        mni_mask_prob,
        0.5,
        1.0,
        1,
        0
    )

    mni_brain = ants.mask_image(
        mni_head,
        mni_mask_bin
    )

    ants.image_write(
        mni_brain,
        _MNI_BRAIN_CACHE
    )

    print(
        f"Saved MNI brain template to: "
        f"{_MNI_BRAIN_CACHE}"
    )

    return mni_brain


# =========================================================
# Single MRI preprocessing
# =========================================================

def run_preprocessing(
    input_nii_path: str,
    output_tensor_path: str
):
    """
    Executes the 6-stage preprocessing pipeline:

    1. Reorientation & FOV Cropping
    2. N4 Bias Field Correction
    3. Skull-Stripping
    4. Affine Registration to MNI152
    5. Voxel Resampling & Tensor Formatting
       (1mm isotropic, 160x192x160)
    6. Brain-masked Z-Score Normalization
    """

    print(
        f"\nProcessing: "
        f"{os.path.basename(input_nii_path)}"
    )

    # -------------------------------------------------------------
    # Step 1: Reorientation & FOV Cropping
    # -------------------------------------------------------------

    img = ants.image_read(
        input_nii_path
    )

    print(
        f"  Input intensity range: "
        f"[{img.min():.2f}, {img.max():.2f}]"
    )

    img = ants.reorient_image2(
        img,
        orientation="RAS"
    )

    img_cropped = ants.crop_image(
        img
    )


    # -------------------------------------------------------------
    # Step 2: B1/N4 Bias Field Correction
    # -------------------------------------------------------------

    img_n4 = ants.n4_bias_field_correction(
        img_cropped,
        shrink_factor=2
    )


    # -------------------------------------------------------------
    # Step 3: Brain Extraction / Skull Stripping
    # -------------------------------------------------------------

    mask_prob = antspynet.brain_extraction(
        img_n4,
        modality="t1"
    )

    # Convert probability mask to binary mask.
    mask_bin = ants.threshold_image(
        mask_prob,
        0.5,
        1.0,
        1,
        0
    )

    brain = ants.mask_image(
        img_n4,
        mask_bin
    )

    voxel_count = int(
        mask_bin.sum()
    )

    print(
        f"  Brain mask voxel count: "
        f"{voxel_count}"
    )

    if voxel_count < 100_000:

        raise RuntimeError(
            f"Brain extraction found almost no brain tissue "
            f"({voxel_count} voxels) "
            f"in {os.path.basename(input_nii_path)}. "
            f"This may indicate that the input is not a raw "
            f"T1w MRI scan."
        )


    # -------------------------------------------------------------
    # Step 4: Spatial Registration to 1mm MNI152 Template
    # -------------------------------------------------------------

    mni_brain = get_mni_brain_template()

    print(
        "  Registering brain to MNI152..."
    )

    reg = ants.registration(
        fixed=mni_brain,
        moving=brain,
        type_of_transform="Affine"
    )

    warped_brain = reg[
        "warpedmovout"
    ]


    # -------------------------------------------------------------
    # Step 5: Crop/Pad to exact SFCN matrix
    # -------------------------------------------------------------

    temp_nii = "_temp_reg.nii.gz"

    ants.image_write(
        warped_brain,
        temp_nii
    )

    subject = tio.Subject(
        mri=tio.ScalarImage(
            temp_nii
        )
    )

    crop_pad = tio.CropOrPad(
        (160, 192, 160)
    )

    processed_subject = crop_pad(
        subject
    )

    data = (
        processed_subject
        .mri
        .data
        .squeeze()
        .numpy()
    )

    final_affine = (
        processed_subject
        .mri
        .affine
    )

    # Remove temporary registered image.
    if os.path.exists(temp_nii):
        os.remove(temp_nii)


    # -------------------------------------------------------------
    # Step 6: Brain-Masked Z-Score Normalization
    # -------------------------------------------------------------

    brain_mask = data > 0

    standardized_brain = np.zeros_like(
        data,
        dtype=np.float32
    )

    if np.any(brain_mask):

        mu = np.mean(
            data[brain_mask]
        )

        sigma = np.std(
            data[brain_mask]
        )

        standardized_brain[brain_mask] = (
            data[brain_mask] - mu
        ) / (sigma + 1e-8)


    # -------------------------------------------------------------
    # Save NPZ
    # -------------------------------------------------------------

    output_directory = os.path.dirname(
        output_tensor_path
    )

    if output_directory:
        os.makedirs(
            output_directory,
            exist_ok=True
        )

    np.savez_compressed(
        output_tensor_path,
        data=standardized_brain
    )


    # -------------------------------------------------------------
    # Save NIfTI for Slicer
    # -------------------------------------------------------------

    slicer_nii_path = (
        output_tensor_path
        .replace(
            ".npz",
            ".nii.gz"
        )
    )

    nii_img = nib.Nifti1Image(
        standardized_brain,
        final_affine
    )

    nib.save(
        nii_img,
        slicer_nii_path
    )


    print(
        f"  Saved training tensor: "
        f"{output_tensor_path}"
    )

    print(
        f"  Saved Slicer volume: "
        f"{slicer_nii_path}"
    )


# =========================================================
# FIND ALL MRI FILES RECURSIVELY
# =========================================================

def find_t1_files(raw_dir: str):
    """
    Recursively searches for all NIfTI MRI files inside raw_dir.

    Example input structure:

    T1/
        OAS30001_MR_d0129/
            anat2/
                sub-OAS30001_ses-d0129_run-1_T1w.nii.gz

    The complete file paths are returned.
    """

    nii_files = []

    # Find .nii files
    nii_files.extend(
        glob.glob(
            os.path.join(
                raw_dir,
                "**",
                "*.nii"
            ),
            recursive=True
        )
    )

    # Find .nii.gz files
    nii_files.extend(
        glob.glob(
            os.path.join(
                raw_dir,
                "**",
                "*.nii.gz"
            ),
            recursive=True
        )
    )

    # Remove unwanted files
    nii_files = [
        f
        for f in nii_files
        if not f.endswith(".json")
        and "_preproc" not in os.path.basename(f)
    ]

    return sorted(nii_files)


# =========================================================
# BATCH PREPROCESSING
# =========================================================

def batch_preprocess_t1(
    raw_dir: str,
    output_base_dir: str
):
    """
    Preprocesses every T1w MRI found anywhere inside raw_dir.

    The original folder structure is preserved.

    Example:

    INPUT:

    T1/
    └── OAS30001_MR_d0129/
        └── anat2/
            └── sub-OAS30001_ses-d0129_run-1_T1w.nii.gz


    OUTPUT:

    T1 preprocessed/
    └── OAS30001_MR_d0129/
        └── anat2/
            ├── sub-OAS30001_ses-d0129_run-1_T1w_preproc.npz
            └── sub-OAS30001_ses-d0129_run-1_T1w_preproc.nii.gz
    """

    # ---------------------------------------------------------
    # Make sure output directory exists
    # ---------------------------------------------------------

    os.makedirs(
        output_base_dir,
        exist_ok=True
    )


    # ---------------------------------------------------------
    # Find all MRI files recursively
    # ---------------------------------------------------------

    raw_files = find_t1_files(
        raw_dir
    )


    print(
        "\n=============================================="
    )

    print(
        "T1 BATCH PREPROCESSING"
    )

    print(
        "=============================================="
    )

    print(
        f"Input directory:"
    )

    print(
        f"  {os.path.abspath(raw_dir)}"
    )

    print(
        f"\nOutput directory:"
    )

    print(
        f"  {os.path.abspath(output_base_dir)}"
    )

    print(
        f"\nFound {len(raw_files)} MRI file(s)"
    )

    print(
        "==============================================\n"
    )


    # ---------------------------------------------------------
    # No files found
    # ---------------------------------------------------------

    if len(raw_files) == 0:

        print(
            "ERROR: No NIfTI MRI files were found."
        )

        print(
            f"Searched recursively inside:"
        )

        print(
            os.path.abspath(raw_dir)
        )

        return []


    # ---------------------------------------------------------
    # Process every MRI
    # ---------------------------------------------------------

    failures = []

    for index, input_file in enumerate(
        raw_files,
        start=1
    ):

        filename = os.path.basename(
            input_file
        )


        # -----------------------------------------------------
        # Calculate the path relative to T1
        # -----------------------------------------------------

        relative_path = os.path.relpath(
            input_file,
            raw_dir
        )


        # -----------------------------------------------------
        # Remove MRI extension
        # -----------------------------------------------------

        if relative_path.endswith(
            ".nii.gz"
        ):

            relative_without_extension = (
                relative_path[:-7]
            )

        elif relative_path.endswith(
            ".nii"
        ):

            relative_without_extension = (
                relative_path[:-4]
            )

        else:

            print(
                f"Skipping unsupported file: "
                f"{input_file}"
            )

            continue


        # -----------------------------------------------------
        # Construct output path
        #
        # This preserves the original folder structure.
        # -----------------------------------------------------

        output_relative_path = (
            relative_without_extension
            + "_preproc.npz"
        )


        output_file = os.path.join(
            output_base_dir,
            output_relative_path
        )


        # -----------------------------------------------------
        # Create corresponding output folder
        # -----------------------------------------------------

        output_folder = os.path.dirname(
            output_file
        )

        os.makedirs(
            output_folder,
            exist_ok=True
        )


        # -----------------------------------------------------
        # Skip already processed files
        # -----------------------------------------------------

        if os.path.exists(
            output_file
        ):

            print(
                f"\n[{index}/{len(raw_files)}] "
                f"SKIPPING"
            )

            print(
                f"  Already exists:"
            )

            print(
                f"  {output_file}"
            )

            continue


        # -----------------------------------------------------
        # Display current file
        # -----------------------------------------------------

        print(
            "\n=============================================="
        )

        print(
            f"[{index}/{len(raw_files)}]"
        )

        print(
            "Input:"
        )

        print(
            f"  {input_file}"
        )

        print(
            "Output:"
        )

        print(
            f"  {output_file}"
        )

        print(
            "=============================================="
        )


        # -----------------------------------------------------
        # Run preprocessing
        # -----------------------------------------------------

        try:

            run_preprocessing(
                input_file,
                output_file
            )

        except Exception as e:

            print(
                "\nFAILED"
            )

            print(
                f"File:"
            )

            print(
                f"  {input_file}"
            )

            print(
                f"Error:"
            )

            print(
                f"  {e}"
            )

            failures.append(
                input_file
            )


    # ---------------------------------------------------------
    # Final report
    # ---------------------------------------------------------

    print(
        "\n\n=============================================="
    )

    print(
        "BATCH PREPROCESSING COMPLETE"
    )

    print(
        "=============================================="
    )

    print(
        f"Total MRI files : {len(raw_files)}"
    )

    print(
        f"Successful      : "
        f"{len(raw_files) - len(failures)}"
    )

    print(
        f"Failed          : {len(failures)}"
    )

    print(
        f"Output location : "
        f"{os.path.abspath(output_base_dir)}"
    )

    print(
        "=============================================="
    )


    # ---------------------------------------------------------
    # Display failures
    # ---------------------------------------------------------

    if failures:

        print(
            "\nFailed files:"
        )

        for f in failures:

            print(
                f"  {f}"
            )

    else:

        print(
            "\nNo preprocessing failures."
        )


    return failures


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    print(
        "Starting batch preprocessing on T1 folder..."
    )

    batch_preprocess_t1(
        raw_dir="/mnt/d/brain age prep and meth/T1",
        output_base_dir="/mnt/d/brain age prep and meth/T1 preprocessed"
    )

I0000 00:00:1789562458.734941  613668 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789562462.227760  613668 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789562474.699191  613668 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/mnt/d/brain age prep and meth/wsl_env_312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IPr

Starting batch preprocessing on T1 folder...

T1 BATCH PREPROCESSING
Input directory:
  /mnt/d/brain age prep and meth/T1

Output directory:
  /mnt/d/brain age prep and meth/T1 preprocessed

Found 3340 MRI file(s)


[1/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30001_MR_d0129/anat2/sub-OAS30001_ses-d0129_run-01_T1w_preproc.npz

[2/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30001_MR_d0129/anat3/sub-OAS30001_ses-d0129_run-02_T1w_preproc.npz

[3/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30001_MR_d0757/anat2/sub-OAS30001_ses-d0757_run-01_T1w_preproc.npz

[4/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30001_MR_d0757/anat3/sub-OAS30001_ses-d0757_run-02_T1w_preproc.npz

[5/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30001_MR_d2430/anat4/sub-OAS30001_ses-d2430_T1w_preproc.npz

[6/3340] SK

W0000 00:00:1789562551.038862  613668 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1789562551.047584  613668 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1789562551.413542  613668 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13153 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5080, pci bus id: 0000:02:00.0, compute capability: 12.0a
I0000 00:00:1789562556.595165  614753 service.cc:153] XLA service 0x7c1774017250 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789562556.595197  614753 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5080, Compute Capability 12.0a

  Brain mask voxel count: 0

FAILED
File:
  /mnt/d/brain age prep and meth/T1/OAS30339_MR_d0066/anat3/sub-OAS30339_ses-d0066_echo-2_run-03_T1w.nii.gz
Error:
  Brain extraction found almost no brain tissue (0 voxels) in sub-OAS30339_ses-d0066_echo-2_run-03_T1w.nii.gz. This may indicate that the input is not a raw T1w MRI scan.

[985/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30339_MR_d0066/anat4/sub-OAS30339_ses-d0066_run-01_T1w_preproc.npz

[986/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30339_MR_d0066/anat5/sub-OAS30339_ses-d0066_run-02_T1w_preproc.npz

[987/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30341_MR_d0025/anat2/sub-OAS30341_ses-d0025_T1w_preproc.npz

[988/3340] SKIPPING
  Already exists:
  /mnt/d/brain age prep and meth/T1 preprocessed/OAS30342_MR_d0001/anat3/sub-OAS30342_ses-d0001_T1w_preproc.npz

[989/3340] SKIPPING
  Already exists:
  /mnt/d/brain 

In [ ]:
# =========================================================
# Batch preprocessing over the full cohort
# =========================================================
# build_manifest() and everything in Section III below need many
# preprocessed scans (the paper uses 1,027 repeat scans / 364+ subjects),
# not just the single test file processed by the __main__ block above.

def batch_preprocess_t1(raw_dir: str = "T1", output_base_dir: str = "T1 preprocessed", pattern: str = "**/*_T1w.nii*"):
    """
    Runs run_preprocessing() over every raw T1w NIfTI found in raw_dir recursively,
    maintaining the directory structure in output_base_dir.
    """
    import glob
    from pathlib import Path
    
    raw_files = sorted(
        f for f in glob.glob(os.path.join(raw_dir, pattern), recursive=True)
        if "_preproc" not in f and not f.endswith(".json")
    )
    print(f"Found {len(raw_files)} raw T1w file(s) in {raw_dir}")

    failures = []
    for f in raw_files:
        # Get relative path to maintain structure
        rel_path = os.path.relpath(f, raw_dir)
        
        # Replace .nii.gz or .nii with _preproc.npz for the output filename
        fname = os.path.basename(f)
        if fname.endswith('.nii.gz'):
            out_name = fname[:-7] + '_preproc.npz'
        elif fname.endswith('.nii'):
            out_name = fname[:-4] + '_preproc.npz'
        else:
            out_name = fname + '_preproc.npz'
        
        # Determine output directory
        rel_dir = os.path.dirname(rel_path)
        out_dir = os.path.join(output_base_dir, rel_dir)
        os.makedirs(out_dir, exist_ok=True)
        
        out_path = os.path.join(out_dir, out_name)
        if os.path.exists(out_path):
            continue
        try:
            run_preprocessing(f, out_path)
        except Exception as e:
            print(f"  FAILED on {os.path.basename(f)}: {e}")
            failures.append(f)

    print(f"Batch preprocessing complete. {len(failures)} failure(s) out of {len(raw_files)}.")
    return failures

In [ ]:
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, max_pool=True):
        super().__init__()
        layers = [
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        ]
        if max_pool:
            layers.append(nn.MaxPool3d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class SFCN(nn.Module):
    """
    Simple Fully Convolutional Network (SFCN) for 3D Brain MRI
    Expects input shape: (B, 1, 160, 192, 160)
    """
    def __init__(self, output_dim=1, dropout_p=0.5):
        super().__init__()
        # 5 feature-extraction blocks downsampling the volume from 160x192x160 down to 5x6x5
        self.block1 = ConvBlock(1, 32, max_pool=True)     # 80 x 96 x 80
        self.block2 = ConvBlock(32, 64, max_pool=True)    # 40 x 48 x 40
        self.block3 = ConvBlock(64, 128, max_pool=True)   # 20 x 24 x 20
        self.block4 = ConvBlock(128, 256, max_pool=True)  # 10 x 12 x 10
        self.block5 = ConvBlock(256, 256, max_pool=True)  # 5 x 6 x 5

        # Block 6: 1x1x1 conv to increase non-linearity
        self.block6 = nn.Sequential(
            nn.Conv3d(256, 64, kernel_size=1, bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True)
        )
        # Block 7: Global Pooling & Linear Projection
        self.global_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(64, output_dim)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x.squeeze(-1)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset

class BrainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        """
        df must contain columns: 'file_path' and 'age'
        """
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Load compressed 160x192x160 tensor
        data = np.load(row['file_path'])['data']
        # Add channel dimension: (1, 160, 192, 160)
        tensor = torch.from_numpy(data).unsqueeze(0).float()
        age = torch.tensor(row['age'], dtype=torch.float32)

        if self.transform:
            tensor = self.transform(tensor)

        return tensor, age

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_mae = 0.0, 0.0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        preds = model(inputs).view_as(targets)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        total_mae += torch.sum(torch.abs(preds - targets)).item()

    return total_loss / len(loader.dataset), total_mae / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_mae = 0.0, 0.0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs).view_as(targets)
            loss = criterion(preds, targets)

            total_loss += loss.item() * inputs.size(0)
            total_mae += torch.sum(torch.abs(preds - targets)).item()

    return total_loss / len(loader.dataset), total_mae / len(loader.dataset)


# ## Section III-A -- Manifest & subject-level split